# MCP SQL Query Expert (GCP-native) — v1

**Companion to `sql_query_expert_v5.1.ipynb`.** Same job — answer natural-language questions from the analytics PostgreSQL DB with a **one-line human answer, never SQL** — but through an **MCP (Model Context Protocol)** architecture instead of the in-process `generate → validate → execute` pipeline.

Run this side-by-side with the SQL Expert pipeline and decide which fits RFP&nbsp;AIQ.

### The stack in this notebook (all GCP-native, no Claude)

| Role | What we use |
|------|-------------|
| **MCP server** | **Google's [MCP Toolbox for Databases](https://github.com/googleapis/genai-toolbox)** — an open-source, GCP-maintained MCP server that connects directly to Cloud SQL for PostgreSQL. This is the *“GCP server MCP tool.”* |
| **Database** | The same Cloud SQL Postgres the SQL Expert uses (`SQL_EXPERT_DB_*` from `.env`). |
| **MCP client / driver** | **Gemini via Vertex AI** (`gemini-2.5-flash`), matching `LLM_PROVIDER=vertexai`. |
| **Agent glue** | `toolbox-langchain` + LangGraph ReAct agent. |

> Nothing in this notebook calls Claude/Anthropic. The whole path stays inside GCP (Cloud SQL + Vertex AI, IAM/ADC auth).

## How MCP differs from the SQL Expert pipeline

```
  This notebook (MCP):                        sql_expert/pipeline.py:

  Gemini (Vertex)                             Gemini/LLM
     | picks a tool + params                     | generates raw SQL
     v                                           v
  MCP client (toolbox-langchain)              validators.py  (allowlist, read-only)
     | MCP over HTTP                             v
     v                                        EXPLAIN dry-run
  MCP Toolbox server  <-- tools.yaml             v
     | pre-authored parameterized SQL         read-only execute (row cap, 15s)
     v                                           v
  Cloud SQL Postgres                          one-line summary
```

**Key philosophical difference.** The pipeline lets the LLM write *arbitrary* SQL and then defends against it (`validators.py` + `EXPLAIN` + row cap + timeout). Google's Toolbox flips that: the LLM can only call **pre-authored, parameterized queries** you wrote — the statement is fixed, only bound parameters vary, so there is **no free-form SQL and no injection surface**. That is *safe-by-construction* rather than *generate-then-validate*.

Both approaches are shown; the trade-offs are summarized at the end.

## Step 0 — Install dependencies

Run once. (The Toolbox *server* binary is downloaded in Step 2 — these are the Python client libraries.)

In [ ]:
%pip install -q toolbox-langchain langchain-google-vertexai langgraph python-dotenv requests

## Step 1 — Config

Reuse the project's `Settings` so the DB and Vertex config come from the **same `.env`** the running backend uses. No new secrets.

Vertex AI uses **ADC** (Application Default Credentials) — no API key. If the Vertex cell later fails auth, run once in a terminal:

```bash
gcloud auth application-default login
```

In [ ]:
import sys, pathlib

# Make the backend_py package importable when running from anywhere.
HERE = pathlib.Path.cwd()
BACKEND = HERE if (HERE / "config.py").exists() else HERE / "backend_py"
sys.path.insert(0, str(BACKEND))

from config import get_settings

settings = get_settings()

# ===================== CHOOSE YOUR MODE HERE =====================
#   "curated"  = Mode 1: you author fixed parameterized queries in tools.yaml.
#                Safe-by-construction (Gemini can only call queries you wrote).
#   "prebuilt" = Mode 2: Toolbox auto-generates generic tools from the DB
#                (list_tables / get_table_schema / free-form execute_sql).
#                Zero authoring, runs against your real schema, but UNGUARDED.
MODE = "curated"
# ================================================================
assert MODE in ("curated", "prebuilt"), MODE

DB = {
    "host": settings.sql_expert_db_host,
    "port": settings.sql_expert_db_port,
    "database": settings.sql_expert_db_name,
    "user": settings.sql_expert_db_user,
    "password": settings.sql_expert_db_password.get_secret_value(),
}
GCP_PROJECT = settings.gcp_project
GCP_LOCATION = settings.gcp_location or "us-central1"
GEMINI_MODEL = settings.vertex_model or "gemini-2.5-flash"
MAX_ROWS = settings.sql_expert_max_rows

assert DB["host"], "SQL_EXPERT_DB_HOST is empty — set SQL_EXPERT_* in .env (same DB the pipeline uses)."
assert GCP_PROJECT, "GCP_PROJECT is empty — required for Vertex AI (Gemini)."

print(f"MODE   : {MODE}")
print(f"DB     : {DB['user']}@{DB['host']}:{DB['port']}/{DB['database']}")
print(f"Vertex : project={GCP_PROJECT} location={GCP_LOCATION} model={GEMINI_MODEL}")
print(f"Row cap: {MAX_ROWS}")

## Step 2 — Download the MCP Toolbox server

Grabs the official Toolbox binary for your OS from Google's public bucket. Alternatives (see the Toolbox README): Docker image `us-central1-docker.pkg.dev/database-toolbox/toolbox/toolbox`, `brew install mcp-toolbox`, or `npx @toolbox-sdk/server`.

> **Read-only recommendation.** Point the Toolbox `postgres` source at a **read-only DB role** in production. This notebook reuses `SQL_EXPERT_DB_USER`; if that user has write grants, create a read-only role and set it here.

In [ ]:
import platform, stat, urllib.request, pathlib

TOOLBOX_VERSION = "1.7.0"  # pin; check the repo Releases for the latest

_sys = platform.system().lower()
_arch = "arm64" if platform.machine().lower() in ("arm64", "aarch64") else "amd64"
if _sys == "windows":
    _os_dir, _fname = "windows/amd64", "toolbox.exe"
elif _sys == "darwin":
    _os_dir, _fname = f"darwin/{_arch}", "toolbox"
else:
    _os_dir, _fname = f"linux/{_arch}", "toolbox"

TOOLBOX_BIN = (BACKEND / _fname).resolve()
url = f"https://storage.googleapis.com/mcp-toolbox-for-databases/v{TOOLBOX_VERSION}/{_os_dir}/{_fname}"

if not TOOLBOX_BIN.exists():
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, TOOLBOX_BIN)
    if _sys != "windows":
        TOOLBOX_BIN.chmod(TOOLBOX_BIN.stat().st_mode | stat.S_IEXEC)
    print("Saved:", TOOLBOX_BIN)
else:
    print("Already present:", TOOLBOX_BIN)

## Step 3 — Author `tools.yaml` (Mode 1 / `curated` only)

**This cell only matters when `MODE = "curated"`.** In `prebuilt` mode the launch cell ignores `tools.yaml` and lets Toolbox generate tools from the DB, so you can skip authoring entirely.

In curated mode, each `tool` is a **fixed, parameterized SQL statement** — Gemini chooses *which* tool and *what parameters*, but can never rewrite the SQL. Two **discovery** tools (`list_tables`, `describe_table`) let it learn the live schema, plus a few **analytics** tools using the split-table convention from `services/sql_expert/table_notes.py` (`bids` + `bids_old` via `UNION ALL`).

> ⚠️ **Adjust column names to your real schema.** These statements assume common columns (`status`). If you don't want to hand-tune queries at all, use `MODE = "prebuilt"` in Step 1 instead — it runs against your real schema with zero authoring (but no safe-by-construction guarantee).

In [ ]:
import yaml, pathlib

SCHEMA = settings.sql_expert_db_schema or "public"

config = {
    "sources": {
        "rfp-analytics": {
            "kind": "postgres",
            "host": DB["host"],
            "port": DB["port"],
            "database": DB["database"],
            "user": DB["user"],
            "password": DB["password"],
        }
    },
    "tools": {
        "list_tables": {
            "kind": "postgres-sql",
            "source": "rfp-analytics",
            "description": "List all base tables available in the analytics database. Call this first to discover what data exists.",
            "statement": (
                "SELECT table_name FROM information_schema.tables "
                f"WHERE table_schema = '{SCHEMA}' AND table_type = 'BASE TABLE' "
                "ORDER BY table_name;"
            ),
        },
        "describe_table": {
            "kind": "postgres-sql",
            "source": "rfp-analytics",
            "description": "Return the columns and data types of one table. Use it to learn a table's shape before answering.",
            "parameters": [
                {"name": "table_name", "type": "string", "description": "Exact table name, e.g. 'bids'."}
            ],
            "statement": (
                "SELECT column_name, data_type FROM information_schema.columns "
                f"WHERE table_schema = '{SCHEMA}' AND table_name = $1 "
                "ORDER BY ordinal_position;"
            ),
        },
        "total_bids": {
            "kind": "postgres-sql",
            "source": "rfp-analytics",
            "description": "Total number of bids across current and historical tables (bids + bids_old).",
            "statement": (
                "SELECT count(*) AS total_bids FROM "
                "(SELECT 1 FROM bids UNION ALL SELECT 1 FROM bids_old) t;"
            ),
        },
        "bids_by_status": {
            "kind": "postgres-sql",
            "source": "rfp-analytics",
            "description": "Count bids with a given status, combining current and historical tables. Example statuses: 'Active', 'Complete', 'Solicitation'.",
            "parameters": [
                {"name": "status", "type": "string", "description": "Bid status to filter on."}
            ],
            "statement": (
                "SELECT count(*) AS n FROM "
                "(SELECT status FROM bids UNION ALL SELECT status FROM bids_old) t "
                "WHERE t.status = $1;"
            ),
        },
        "total_suppliers": {
            "kind": "postgres-sql",
            "source": "rfp-analytics",
            "description": "Total number of suppliers across current and historical tables (suppliers + suppliers_old).",
            "statement": (
                "SELECT count(*) AS total_suppliers FROM "
                "(SELECT 1 FROM suppliers UNION ALL SELECT 1 FROM suppliers_old) t;"
            ),
        },
    },
    "toolsets": {
        "rfp-aiq": [
            "list_tables",
            "describe_table",
            "total_bids",
            "bids_by_status",
            "total_suppliers",
        ]
    },
}

TOOLS_YAML = BACKEND / "tools.yaml"
TOOLS_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print("Wrote", TOOLS_YAML)
print("(password is redacted below)")
print(yaml.safe_dump({**config, "sources": {"rfp-analytics": {**config['sources']['rfp-analytics'], 'password': '***'}}}, sort_keys=False))

## Step 4 — Launch the Toolbox MCP server

Starts the server as a background process. It serves the tools over HTTP on `:5000` (MCP endpoint at `/mcp`). We leave it running for the rest of the notebook; the final cell terminates it.

In [ ]:
import os, subprocess, time, socket

TOOLBOX_PORT = 5000
TOOLBOX_URL = f"http://127.0.0.1:{TOOLBOX_PORT}"

def _port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

# Stop a previous run started by this notebook.
proc = globals().get("proc")
if isinstance(proc, subprocess.Popen) and proc.poll() is None:
    proc.terminate()
    time.sleep(1)

base_args = [str(TOOLBOX_BIN), "--address", "127.0.0.1", "--port", str(TOOLBOX_PORT)]
env = os.environ.copy()

if MODE == "curated":
    # Mode 1: serve the fixed parameterized queries you authored in tools.yaml.
    args = base_args + ["--config", str(TOOLS_YAML)]
    TOOLSET_NAME = "rfp-aiq"
else:
    # Mode 2: Toolbox auto-generates a generic postgres toolset from the DB.
    # Connection comes from POSTGRES_* env vars (no tools.yaml needed).
    args = base_args + ["--prebuilt", "postgres"]
    env.update({
        "POSTGRES_HOST": DB["host"],
        "POSTGRES_PORT": str(DB["port"]),
        "POSTGRES_DATABASE": DB["database"],
        "POSTGRES_USER": DB["user"],
        "POSTGRES_PASSWORD": DB["password"],
    })
    TOOLSET_NAME = None  # load all auto-generated tools

proc = subprocess.Popen(args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(30):
    if _port_open(TOOLBOX_PORT):
        break
    if proc.poll() is not None:
        raise RuntimeError("Toolbox exited early:\n" + proc.stdout.read())
    time.sleep(0.5)
else:
    raise RuntimeError("Toolbox did not start listening on :%d" % TOOLBOX_PORT)

print(f"✓ Toolbox MCP server ({MODE}) at {TOOLBOX_URL}  (MCP endpoint: {TOOLBOX_URL}/mcp)")

## Step 5 — Load the tools into a Gemini (Vertex) agent

`toolbox-langchain` fetches the tool definitions from the running MCP server and returns LangChain tools. We hand them to a LangGraph ReAct agent backed by `ChatVertexAI` (Gemini). The system prompt pins the **same output contract** as the SQL Expert: *one human sentence, never SQL*.

In [ ]:
from toolbox_langchain import ToolboxClient
from langchain_google_vertexai import ChatVertexAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage

# Load tools from the running MCP server. In curated mode we load our named
# toolset; in prebuilt mode we load all auto-generated tools. Method names have
# shifted across toolbox-langchain versions, so try the known variants.
tb = ToolboxClient(TOOLBOX_URL)

async def _load(name):
    for attr in ("aload_toolset", "load_toolset"):
        fn = getattr(tb, attr, None)
        if fn is None:
            continue
        res = fn() if name is None else fn(name)
        return await res if hasattr(res, "__await__") else res
    raise RuntimeError("ToolboxClient has no (a)load_toolset method")

tools = await _load(TOOLSET_NAME)
print("Loaded MCP tools:", [t.name for t in tools])

llm = ChatVertexAI(model=GEMINI_MODEL, project=GCP_PROJECT, location=GCP_LOCATION, temperature=0)

SYSTEM = (
    "You are RFP AIQ, the Sysco Bid Intelligence assistant. Answer the user's question "
    "by calling the provided database tools. If you need to know what tables or columns "
    "exist, use the discovery tools first. "
    "Reply with ONE short, human sentence stating the answer. Never show SQL, tool names, "
    "JSON, or row dumps. If the tools cannot answer, say so in one sentence."
)

agent = create_react_agent(llm, tools)
print("✓ Gemini agent ready")

## Step 6 — Ask questions

`ask()` prints the tool calls Gemini made (so you can see the MCP round-trips) and the final one-line answer.

In [ ]:
async def ask(question: str) -> str:
    result = await agent.ainvoke(
        {"messages": [SystemMessage(content=SYSTEM), HumanMessage(content=question)]}
    )
    msgs = result["messages"]
    # Show which MCP tools were invoked.
    for m in msgs:
        for tc in getattr(m, "tool_calls", None) or []:
            print(f"   → tool: {tc['name']}({tc.get('args', {})})")
    answer = msgs[-1].content
    print(f"Q: {question}\nA: {answer}\n")
    return answer

await ask("What tables are available?")
await ask("How many bids are there in total?")
await ask("How many bids have the status Complete?")
await ask("How many suppliers do we have?")

## Step 7 — The two modes, recap

You already control which mode runs via **`MODE` in Step 1**:

| | `MODE = "curated"` (Mode 1) | `MODE = "prebuilt"` (Mode 2) |
|---|---|---|
| Who writes the SQL | **You** (fixed queries in `tools.yaml`) | **Toolbox** auto-generates generic tools incl. free-form `execute_sql` |
| Schema tuning | Match column names to your DB | None — runs against the live schema as-is |
| Safety | Safe-by-construction (no arbitrary SQL) | **Unguarded** — the model can run any SQL |
| `sql_expert/validators.py` guardrails | N/A (SQL is pre-authored) | **Not applied** — `execute_sql` bypasses them |

> If you want arbitrary LLM-generated SQL *with* the read-only / allowlist / `EXPLAIN` / row-cap protections, don't expose `prebuilt`'s `execute_sql` directly — keep the `sql_expert` pipeline in front of the DB.

## Step 8 — MCP Toolbox vs the SQL Expert pipeline

| Dimension | `services/sql_expert` pipeline | MCP Toolbox + Gemini (this notebook) |
|---|---|---|
| **SQL origin** | LLM generates raw SQL per question | Pre-authored parameterized queries in `tools.yaml` |
| **Safety model** | Generate → validate → `EXPLAIN` → read-only → row cap → timeout | Safe-by-construction: no free-form SQL, no injection surface |
| **New question support** | Automatic — any answerable question works | Add a new tool to `tools.yaml` (or expose `execute_sql`, losing the guarantee) |
| **Guardrail coverage** | Every query passes `validators.py` | Guaranteed only for parameterized tools; `execute_sql` is unguarded |
| **Infrastructure** | In-process, no extra service | Separate long-running server to deploy/monitor |
| **Reuse across hosts** | App-internal only | Any MCP host — Gemini, Claude Desktop, Cursor, IDEs |
| **Driver LLM** | Any (`LLM_PROVIDER`) | Gemini on Vertex here; server is model-agnostic |

### Recommendation for RFP AIQ

- **Keep the `sql_expert` pipeline** as the in-app AIQ chat backend: it already answers *open-ended* questions safely, needs no extra service, and its validators cover LLM-generated SQL. MCP adds a moving part (a server to run) without changing that answer.
- **Reach for Google's MCP Toolbox** when you want the *same DB* usable by **external MCP hosts** — e.g. letting Claude Desktop, Cursor, or a Gemini agent query live bid data during development, or exposing a **curated, safe-by-construction** tool surface to a broader set of agents. The parameterized-tool model is an excellent fit there precisely because it never lets a host run arbitrary SQL.

In short: **pipeline for the product's chat, MCP Toolbox for cross-host / agent access.**

## Cleanup — stop the Toolbox server

In [ ]:
if isinstance(proc, subprocess.Popen) and proc.poll() is None:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("Toolbox server stopped.")
else:
    print("Toolbox server was not running.")